# Séance 7 · Analyser : enquêter sur des données · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Des données propres, c'est bien. Mais un tableau de 800 lignes ne dit rien à personne. Aujourd'hui tu mènes l'**enquête** : tu poses les bonnes questions aux données, tu repères une tendance, une anomalie, une corrélation, et tu apprends à ne pas te faire piéger (corrélation n'est pas causalité, graphiques menteurs, données biaisées). Raconter ce que tu as trouvé à un client, ce sera la séance 8.

Ce notebook tourne dans **Google Colab** (rien à installer). Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter.

**Livrable de la séance** : ton enquête, trois questions, trois réponses chiffrées, trois graphiques, et pour chacune le piège que tu as vérifié.

## Préparation

Deux jeux de données : **Tips** (244 additions d'un restaurant : montant, pourboire, jour, nombre de convives) et les **Pokémon** de la séance 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    tips = pd.read_csv(URL_TIPS)
    pokemon = pd.read_csv(URL_POKEMON)
    print("Tips :", tips.shape, "· Pokémon :", pokemon.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)
tips.head()

In [ ]:
# Vérification des exercices : ✅ si c'est juste, ❌ sinon, sans jamais donner la réponse.
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok

## 1. L'analyse exploratoire : quelles questions poser ?

L'**analyse exploratoire** (EDA en anglais), c'est faire connaissance avec les données avant de conclure quoi que ce soit. Comme un détective qui arrive sur une scène : on regarde tout, on ne juge rien, on note ce qui est bizarre.

Une bonne question d'analyse se reconnaît à trois choses : elle est **précise** (« les pourboires sont-ils plus gros le week-end ? » plutôt que « c'est quoi les pourboires ? »), on peut y **répondre avec les colonnes qu'on a**, et la réponse **intéresse quelqu'un** (le patron du restaurant, ici).

Le réflexe de base : `describe()` pour les nombres, `value_counts()` pour les catégories.

In [ ]:
print(tips.describe().round(2))
print()
print(tips["day"].value_counts())

**Exercice** : écris 3 questions précises sur le dataset Tips (dans un commentaire), puis réponds à la première avec `groupby`. Exemples de départ : le pourboire moyen change-t-il selon le jour ? Les fumeurs laissent-ils plus ? Les grandes tables donnent-elles plus par personne ?

In [ ]:
# À toi

<details><summary>Solution</summary>

```python
# Q1 : le pourboire moyen change-t-il selon le jour ?
# Q2 : les fumeurs laissent-ils un pourboire plus gros ?
# Q3 : le pourboire par personne baisse-t-il avec la taille de la table ?
print(tips.groupby("day")["tip"].mean().round(2))
print(tips.groupby("smoker")["tip"].mean().round(2))
tips["tip_par_personne"] = tips["tip"] / tips["size"]
print(tips.groupby("size")["tip_par_personne"].mean().round(2))
```
</details>

## 2. Repérer une tendance

Une **tendance**, c'est une direction générale : « plus l'addition est grosse, plus le pourboire est gros ». Le graphique roi pour ça : le **nuage de points** (*scatter*). Seaborn ajoute en une ligne la droite qui résume la tendance.

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.regplot(data=tips, x="total_bill", y="tip", scatter_kws={"alpha": 0.5})
plt.xlabel("Addition ($)"); plt.ylabel("Pourboire ($)")
plt.title("Plus l'addition est grosse, plus le pourboire monte")
plt.show()

In [ ]:
# Une tendance par catégorie : le pourboire moyen selon le jour et le moment
tendance = tips.groupby(["day", "time"])["tip"].mean().round(2).unstack()
print(tendance)
tendance.plot(kind="bar", figsize=(7, 4), title="Pourboire moyen par jour et moment")
plt.ylabel("Pourboire ($)"); plt.show()

**Exercice** : sur les Pokémon, y a-t-il une tendance entre `Attack` et `Sp. Atk` ? Trace un `regplot`. Puis regarde la tendance du `Total` moyen par `Generation` avec `groupby` et un graphique en barres.

In [ ]:
# À toi

<details><summary>Solution</summary>

```python
plt.figure(figsize=(7, 4.5))
sns.regplot(data=pokemon, x="Attack", y="Sp. Atk", scatter_kws={"alpha": 0.4})
plt.show()
pokemon.groupby("Generation")["Total"].mean().plot(kind="bar", title="Puissance moyenne par génération")
plt.show()
```
</details>

## 3. Repérer une anomalie

Une **anomalie** (ou *outlier*), c'est une valeur qui sort du lot : un pourboire de 10 $ sur une addition de 7 $, un Pokémon avec 255 points de vie. Ce n'est pas forcément une erreur : parfois c'est **la** chose intéressante à raconter.

L'outil visuel : la **boîte à moustaches** (*boxplot*). La boîte contient la moitié centrale des valeurs, les points isolés au-delà des moustaches sont les suspects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=tips, x="day", y="tip", ax=axes[0])
axes[0].set_title("Pourboires par jour : les points isolés sont les anomalies")
sns.boxplot(data=pokemon, y="HP", ax=axes[1])
axes[1].set_title("Points de vie des Pokémon")
plt.show()

In [ ]:
# La règle classique : anomalie = au-delà de 1,5 fois l'écart entre le 1er et le 3e quart
q1, q3 = pokemon["HP"].quantile([0.25, 0.75])
limite_haute = q3 + 1.5 * (q3 - q1)
anomalies = pokemon[pokemon["HP"] > limite_haute]
print("Limite :", limite_haute, "PV ·", len(anomalies), "Pokémon au-dessus")
anomalies[["Name", "Type 1", "HP", "Total"]].sort_values("HP", ascending=False).head()

**Exercice** : trouve les anomalies de pourboire dans Tips avec la même règle sur la colonne `tip`. Qui a laissé le plus gros pourboire, et sur quelle addition ? Calcule aussi le pourcentage (`tip / total_bill`) : l'anomalie est-elle la même ? Range le nombre d'anomalies dans `nb_anomalies`.

In [ ]:
# À toi
nb_anomalies = None   # ← le nombre de pourboires au-dessus de la limite

In [ ]:
verifier("Nombre d'anomalies", lambda: nb_anomalies == (tips["tip"] > np.percentile(tips["tip"], 75) + 1.5 * (np.percentile(tips["tip"], 75) - np.percentile(tips["tip"], 25))).sum())

<details><summary>Solution</summary>

```python
q1, q3 = tips["tip"].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
nb_anomalies = (tips["tip"] > limite).sum()
print(tips[tips["tip"] > limite].sort_values("tip", ascending=False))
tips["pourcentage"] = (tips["tip"] / tips["total_bill"] * 100).round(1)
print(tips.sort_values("pourcentage", ascending=False).head(3))   # un pourboire de 71 % sur une addition de 7 $ !
```
</details>

## 4. Repérer une corrélation

Une **corrélation** mesure à quel point deux colonnes bougent ensemble, entre **-1** (quand l'une monte, l'autre descend toujours) et **+1** (elles montent toujours ensemble) ; **0** = aucun lien. `df.corr()` calcule toutes les paires d'un coup, et une **heatmap** (carte de chaleur) les colorie pour lire d'un coup d'œil.

In [ ]:
colonnes_num = ["total_bill", "tip", "size"]
print(tips[colonnes_num].corr().round(2))

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
plt.figure(figsize=(7, 5.5))
sns.heatmap(pokemon[stats].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Corrélations entre les stats des Pokémon")
plt.show()

**Exercice** : quelles sont les deux stats les plus corrélées ? Et les deux les moins corrélées ? Vérifie avec un nuage de points pour chacune. (Indice : `Speed` et `Defense`, ça s'explique bien avec des Pokémon : un tank lent ou un rapide fragile ?) Range les deux paires dans `paire_plus_correlee` et `paire_moins_correlee` (deux noms de stats chacune).

In [ ]:
# À toi
paire_plus_correlee = None    # ← les deux stats les plus corrélées
paire_moins_correlee = None   # ← les deux stats les moins corrélées

In [ ]:
_m = pokemon[stats].corr().to_numpy(); _i, _j = np.triu_indices(len(stats), k=1); _v = _m[_i, _j]
verifier("Paire la plus corrélée", lambda: set(paire_plus_correlee) == {stats[_i[_v.argmax()]], stats[_j[_v.argmax()]]})
verifier("Paire la moins corrélée", lambda: set(paire_moins_correlee) == {stats[_i[_v.argmin()]], stats[_j[_v.argmin()]]})

<details><summary>Solution</summary>

```python
c = pokemon[stats].corr()
paires = c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack().dropna().sort_values()
paire_moins_correlee, paire_plus_correlee = paires.index[0], paires.index[-1]
print("Moins corrélées :", paires.index[0], round(paires.iloc[0], 2))
print("Plus corrélées :", paires.index[-1], round(paires.iloc[-1], 2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pokemon.plot.scatter(x=paires.index[-1][0], y=paires.index[-1][1], alpha=0.4, ax=axes[0])
pokemon.plot.scatter(x=paires.index[0][0], y=paires.index[0][1], alpha=0.4, ax=axes[1])
plt.show()
```
</details>

## 5. Ne pas se faire piéger

### 5a. Corrélation n'est pas causalité

En été, les ventes de glaces montent... et les noyades aussi. Les glaces provoquent-elles les noyades ? Non : la **chaleur** fait les deux. On appelle ça une **variable cachée** (ou *facteur de confusion*). Une corrélation dit « ça bouge ensemble », jamais « l'un cause l'autre ».

In [ ]:
# On fabrique des données où la température cause TOUT, et où glaces et noyades ne se parlent jamais
rng = np.random.default_rng(0)
temperature = rng.uniform(5, 35, 120)                                  # 120 journées, de 5 à 35 °C
glaces = 20 + 8 * temperature + rng.normal(0, 25, 120)                 # ventes de glaces
noyades = 0.5 + 0.12 * temperature + rng.normal(0, 0.8, 120)           # noyades (fictives !)
ete = pd.DataFrame({"temperature": temperature.round(1), "glaces": glaces.round(0), "noyades": noyades.clip(0).round(1)})

La corrélation entre glaces et noyades est impressionnante…

In [ ]:
round(ete["glaces"].corr(ete["noyades"]), 2)

…et pourtant, colorier chaque journée par sa température montre que c'est la chaleur qui fait tout.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(ete["glaces"], ete["noyades"], alpha=0.6); axes[0].set_xlabel("Glaces vendues"); axes[0].set_ylabel("Noyades")
axes[0].set_title("Les glaces causent les noyades ?!")
sc = axes[1].scatter(ete["glaces"], ete["noyades"], c=ete["temperature"], cmap="coolwarm", alpha=0.8)
axes[1].set_title("Non : la couleur (température) explique tout"); plt.colorbar(sc, label="°C")
plt.show()

### 5b. Les graphiques trompeurs

Le même chiffre peut avoir l'air d'une catastrophe ou d'un détail selon la façon de le dessiner. Le piège numéro 1 : **l'axe vertical tronqué** (qui ne part pas de zéro). Regarde les deux graphiques ci-dessous : mêmes données.

In [ ]:
mois = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin"]
abonnes = [1020, 1035, 1028, 1050, 1062, 1071]   # abonnés d'une chaîne, +5 % en 6 mois

Les mêmes chiffres, dessinés deux fois : à gauche l'axe part de 1 000, à droite il part de 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(mois, abonnes, color="tab:red"); axes[0].set_ylim(1000, 1080)
axes[0].set_title("TROMPEUR : « la chaîne explose ! » (axe qui part de 1000)")
axes[1].bar(mois, abonnes, color="tab:blue"); axes[1].set_ylim(0, 1200)
axes[1].set_title("HONNÊTE : +5 % en 6 mois (axe qui part de 0)")
for ax in axes: ax.set_ylabel("Abonnés")
plt.show()

### 5c. Les biais dans les données

Un **biais**, c'est quand les données ne représentent pas la réalité qu'on croit mesurer. Le plus courant : l'**échantillon non représentatif**. Exemple : pour savoir combien de personnes jouent aux jeux vidéo, tu fais ton sondage... à la sortie d'un magasin de jeux vidéo. Ton chiffre sera énorme, et faux.

In [ ]:
# 1000 personnes : 40 % jouent vraiment. On simule deux façons de sonder.
rng = np.random.default_rng(1)
joue = rng.random(1000) < 0.40
sortie_magasin = rng.random(1000) < np.where(joue, 0.8, 0.05)   # un joueur a 80 % de chance d'y être, un non-joueur 5 %

echantillon_hasard = rng.choice(joue, 100, replace=False)
echantillon_magasin = joue[sortie_magasin][:100]
print("Vraie proportion de joueurs      :", joue.mean() * 100, "%")
print("Sondage au hasard (100 personnes)     :", echantillon_hasard.mean() * 100, "%")
print("Sondage devant le magasin (100)  :", echantillon_magasin.mean().round(2) * 100, "%  ← biaisé !")

**Exercice** : pour chaque affirmation, dis dans un commentaire quel piège se cache (causalité ? graphique ? biais ?), puis vérifie la troisième avec le code.

1. « Les élèves qui ont un ordinateur portable ont de meilleures notes : donnons un portable à tout le monde ! »
2. « Notre appli est passée de 4,1 à 4,3 étoiles : le graphique montre une barre deux fois plus haute. »
3. « 95 % des avis sur notre jeu sont positifs » (seuls les joueurs qui ont fini le jeu ont été invités à noter).

In [ ]:
# À toi

<details><summary>Solution</summary>

```python
# 1. Causalité : variable cachée probable (revenu du foyer, cadre de travail...)
# 2. Graphique trompeur : axe tronqué, 4,1 → 4,3 c'est +5 %, pas +100 %
# 3. Biais d'échantillon : ceux qui ont abandonné le jeu (probablement mécontents) ne sont pas interrogés
rng = np.random.default_rng(2)
a_fini = rng.random(1000) < 0.5
content = rng.random(1000) < np.where(a_fini, 0.95, 0.30)
print("Avis positifs parmi ceux qui ont fini :", content[a_fini].mean().round(2) * 100, "%")
print("Avis positifs chez tous les joueurs   :", content.mean().round(2) * 100, "%")
```
</details>

## 6. Projet : ton enquête

Tu es data scientist, et ton client est soit le **patron du restaurant** (dataset Tips), soit le **créateur d'un jeu Pokémon** (dataset Pokémon), soit quelqu'un d'autre si tu as ton propre dataset de la séance 2. Il veut savoir trois choses. Ton travail : des réponses chiffrées, et vérifiées.

Consignes :
1. Écris **3 questions précises** qui l'intéressent (section 1).
2. Pour chacune : une cellule de calcul (`groupby`, `corr`, une règle d'anomalie…), le chiffre qui répond, et un graphique avec un titre qui donne la réponse (pas « Pourboire vs addition » mais « Le pourboire suit l'addition, sauf le vendredi »).
3. Pour chacune, vérifie un piège : une corrélation présentée comme une cause ? un axe qui ne part pas de zéro ? un échantillon pas représentatif ?
4. Remplis la fiche d'enquête : elle servira à la séance 8, pour en faire un tableau de bord et un pitch.

In [ ]:
# Question 1 : (écris-la ici en commentaire, puis le calcul)

In [ ]:
# Question 1 : le graphique

In [ ]:
# Question 2 : le calcul

In [ ]:
# Question 2 : le graphique

In [ ]:
# Question 3 : le calcul

In [ ]:
# Question 3 : le graphique

La fiche d'enquête : une ligne par question, avec le chiffre et le piège vérifié. Garde-la, la séance 8 en part.

In [ ]:
enquete = {
    "client": "...",
    "question_1": "... → réponse chiffrée → piège vérifié",
    "question_2": "... → réponse chiffrée → piège vérifié",
    "question_3": "... → réponse chiffrée → piège vérifié",
}
for cle, texte in enquete.items():
    print(f"{cle:12s} {texte}")

## À retenir

- L'**analyse exploratoire** commence par des questions précises, puis `describe()`, `value_counts()`, `groupby()`
- **Tendance** → nuage de points et droite (`regplot`) ; **anomalie** → boîte à moustaches et règle des 1,5 écarts ; **corrélation** → `corr()` et heatmap
- **Corrélation n'est pas causalité** : cherche la variable cachée (la chaleur derrière les glaces et les noyades)
- Un graphique peut mentir : **l'axe vertical doit partir de zéro** pour des barres
- Un **échantillon biaisé** donne un chiffre précis… et faux
- Une réponse d'enquête = un chiffre, un graphique, et un piège vérifié

## Pour montrer aux autres

Pendant les 15 dernières minutes, chacun présente sa meilleure question d'enquête. Trois questions guides :

1. Quelle est ta question, et quel chiffre y répond ?
2. Quel graphique le montre, et pourquoi celui-là ?
3. Quel piège as-tu vérifié avant de conclure (causalité, axe, biais) ?

## Liens gratuits

- Galerie Seaborn : https://seaborn.pydata.org/examples/index.html
- Corrélations absurdes (pour rire, et réfléchir) : https://www.tylervigen.com/spurious-correlations